# Step 1: Prepare features

Ce notebook prépare les attributs au cas par cas:
- traite la connectivité
- complète l'attribut vitesse OSM
- merge les fichiers de source différente: VD, FR, GE pour obtenir 1 fichier GG

In [1]:
import importlib
import geopandas as gpd
import pandas as pd
from functools import reduce
import osmnx as ox
from shapely.geometry import Point, box, LineString
import os
import folium
from folium import Choropleth, CircleMarker, GeoJson
import branca.colormap as cm
from IPython.display import display
pd.set_option('display.max_columns', None)

import network_connectivity
import network_slope
importlib.reload(network_connectivity)
importlib.reload(network_slope)

from network_connectivity import *
from network_slope import *



## Imports

#### Import des segments

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

print("Set parameters")
territory = 'GG' # GG or GE or OTC (need adaptation in attribute table, for later)
network = "walk"  # walk or bike


input_file_path = '../../Data/input'
output_step1_path=f"../../Data/output/{network}/{territory}/step-1"
output_step2_path=f"../../Data/output/{network}/{territory}/step-2"
output_step3_path=f"../../Data/output/{network}/{territory}/step-3"
save_path = f"../../Data/output/{network}/{territory}/step-1"
include = f"include_in_index_{territory}"

save_filtered_attributes = True

# Load segments GeoDataFrame (with 'segment_id')
print("Loading segments...")
# reLoad pedestrian segments
segmented_net = gpd.read_parquet(os.path.join(output_step1_path, "step1_all_segments.parquet"))
segmented_net = segmented_net.to_crs(operation_crs)

# Define a function to save the filtered data
def save(save_filtered_attributes, row, gdf, attribute):
    # Clean geometries first
    gdf["geometry"] = gdf["geometry"].apply(lambda geom: geom.buffer(0) if geom is not None and not geom.is_valid else geom)
    print("Geometries cleaned")

    if not save_filtered_attributes:
        print("Note : Save option is disabled.")
        return

    # Parse save formats: comma-separated list allowed
    raw = str(row.get('save_format', '') or '')
    formats = [f.strip().lower() for f in raw.split(',') if f.strip()]
    if not formats:
        formats = ['csv']  # default fallback

    for fmt in formats:
        try:
            if fmt == 'parquet':
                dirpath = f'{output_step2_path}/parquet_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_parquet(f"{dirpath}/{attribute}.parquet")
                print(f"Filtered data saved for attribute: {attribute} in format: parquet")

            elif fmt == 'gpkg' or fmt == 'geopackage':
                dirpath = f'{output_step2_path}/gpkg_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_file(f"{dirpath}/{attribute}.gpkg", driver="GPKG")
                print(f"Filtered data saved for attribute: {attribute} in format: gpkg")

            elif fmt == 'csv':
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                # to_csv may not preserve geometry consistently; keep original behavior
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Filtered data saved for attribute: {attribute} in format: csv")

            else:
                # Unknown format -> fallback to csv and warn
                dirpath = f'{output_step2_path}/csv_attributs'
                os.makedirs(dirpath, exist_ok=True)
                gdf.to_csv(f"{dirpath}/{attribute}.csv", index=False)
                print(f"Warning: Unknown save format '{fmt}' for attribute {attribute}. Data saved as csv.")
        except Exception as e:
            print(f"Error saving {attribute} as {fmt}: {e}")

Set parameters
Loading segments...


In [3]:
# Pour OTC : créer colonne infra_bike depuis TYPE_AMENA (équivalent de la classification OSM Step 0)
if territory == 'OTC':
    TYPE_AMENA_TO_INFRA_BIKE = {
        'Piste':          'piste_cyclable',
        'Bande':          'bande_cyclable',
        'Voie Bus':       'voie_bus',
        'Dérogation 2R':  'sur_chaussee',
        'Sas':            'sas_velo',
        'Contresens':     'contresens',
    }
    if 'TYPE_AMENA' in segmented_net.columns:
        segmented_net['infra_bike'] = segmented_net['TYPE_AMENA'].map(TYPE_AMENA_TO_INFRA_BIKE).fillna('inconnu')
    else:
        segmented_net['infra_bike'] = 'inconnu'
    print("OTC — infra_bike créé depuis TYPE_AMENA :")
    print(segmented_net['infra_bike'].value_counts())

#### Import des attributs

In [4]:
attributs_info = pd.read_excel(f"{input_file_path}/attributs/attributs_info_{network}_urbain.xlsx", sheet_name="attributs_info")
attributs_info = attributs_info.loc[~attributs_info[include].isin(["False", "Unavailable"])].copy()  # Filter out rows where include_in_index is False or 'Unavailable'

attributs_info

,Class,meta_attribute,attribute,include_in_index_GG,include_in_index_GE,source_type_GE,source_path_GE,file_name_GE,source_type_VD,source_path_VD,file_name_VD,source_type_FR,source_path_FR,file_name_FR,source_type_GG,source_path_GG,file_name_GG,merge_rule_GG,initial_weight,class_weight,buffer_size_GG,buffer_size_GE,impact_attribut,geometry_type_GE,how_GE,value_column_GE,geometry_type_GG,how_GG,value_column_GG,clip_GG,clip_GE,missing_value_strategy,filter_column,filter_values,crs,save_format,Commentaire
0,Attractivité,lac_cours_deau,lac_cours_deau,True,True,sitg,attributs/GE/lac_cours_deau,CAD_NATURE_SOL-SHP/CAD_NATURE_SOL.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.50,50,50,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
1,Attractivité,fontaine,fontaine,True,True,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.50,50,50,favorable,point,count,NaN,point,count,NaN,2.00,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok
2,Attractivité,espaces_ouverts,espaces_ouverts,True,True,sitg,attributs/GE/espaces_ouverts,OBS_EQUIPEMENTS_ESPACES_PUB-SHP/OBS_EQUIPEMENT...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.50,1,1,favorable,polygon,presence,NaN,polygon,presence,NaN,NaN,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
3,Attractivité,amenite,rez_actif,True,True,sitg,attributs/GE/rez_actif,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.50,50,50,favorable,point,count,NaN,point,count,NaN,4.00,10.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
4,Attractivité,transport_public,tp,True,True,sitg,attributs/GE/tp,TPG_ARRETS-SHP/TPG_ARRETS.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.50,0.99,150,150,favorable,point,presence,NaN,point,presence,NaN,NaN,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
5,Attractivité,amenite,amenite,True,True,sitg,attributs/GE/amenite,REG_ENTREPRISE_ETABLISSEMENT-SHP/REG_ENTREPRIS...,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.50,75,75,favorable,point,count,NaN,point,count,NaN,2.00,6.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
8,Commodité,temperature,temperature,True,True,sitg,attributs/GE/temperature,CLIMAT_TEMPERATURE_14H00_P1_2020/CLIMAT_TEMPER...,NaN,NaN,NaN,NaN,NaN,NaN,opendata,attributs/GG/temperature,LST_GrandGeneve_2023.tif,NaN,0.25,0.50,100,250,defavorable,point,raster,temperature,point,raster,temperature,40.00,40.0,keep_nan,filtered,1,2056,"parquet, csv, gpkg",TB
9,Commodité,conflit_md,conflit_md,True,True,sitg,attributs/GE/network_couche_OCT.shp,RP_final_25112025.shp,NaN,NaN,NaN,NaN,NaN,NaN,osm,attributs/GG/osm,osm_attributes.gpkg,NaN,0.25,0.50,1,1,defavorable,line,mean,Partage_us_score,point,sum,severity_score,72.00,2.0,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",TB
10,Commodité,environnement_ouvert_naturel,environnement_ouvert_naturel,True,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/environnement_ouvert_naturel,SIPV_MN_CARTO_GG.gpkg,NaN,0.00,0.00,30,30,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,0.99,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",NaN
11,Commodité,canopee,canopee,True,True,sitg,attributs/GE/canopee,SIPV_ICA_MNC_2023-SHP/SIPV_ICA_MNC_2023.shp,NaN,NaN,NaN,NaN,NaN,NaN,sitg,attributs/GG/canopee,SIPV_MN_CARTO_GG.gpkg,NaN,0.50,0.99,30,30,favorable,polygon,area_ratio,NaN,polygon,area_ratio,NaN,0.99,NaN,zero_if_missing,filtered,1,2056,"parquet, csv, gpkg",ok


## Connectivité du réseau

In [5]:
# Initialize
attribute = None
row = None
gdf = None
print("Data initialized")

# Load
###----change here-----
attribute = 'connectivite'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

# Ensure geometry column is set to avoid spatial index errors
segmented_net = segmented_net.set_geometry("geometry")
print(segmented_net.columns)

# 1. Ajouter u, v, key si besoin
segmented_net = add_uv_columns(segmented_net)

# 2. Calculer les métriques et le score d’alternatives
segmented_net_index = compute_connectivity_metrics(
    segmented_net,
    buffer_m=35,
    compute_betweenness=False,
    betweenness_k=None,
    crs_meter_epsg=2056,
    main_metrics_only=False,
    conn_index_metric="conn_branching_in_buffer",
)

segmented_net_index['filtered'] = 1


# Preview
segmented_net_index.to_crs(target_crs).head()

Data initialized
Processing attribute: connectivite
Index(['u', 'v', 'key', 'walk_class', 'source', 'highway', 'osm_id', 'id',
       'element', 'footway', 'sidewalk', 'name', 'maxspeed', 'service',
       'access', 'foot', 'tracktype', 'surface', 'smoothness', 'len_m',
       'walk_role', 'proxy_dedicated_covered_len_m',
       'proxy_dedicated_coverage_ratio', 'edge_id', 'length', 'geometry',
       'segment_id'],
      dtype='object')


,u,v,key,walk_class,source,highway,osm_id,id,element,footway,sidewalk,name,maxspeed,service,access,foot,tracktype,surface,smoothness,len_m,walk_role,proxy_dedicated_covered_len_m,proxy_dedicated_coverage_ratio,edge_id,length,geometry,segment_id,length_m,conn_deadend_flag,conn_intersection_flag,conn_nodes_in_buffer,conn_intersections_in_buffer,conn_branching_in_buffer,conn_index_score,filtered
0,-3671742797194614437,-9140018067926386715,000000,walk_dedicated,OSM_highway_dedicated,footway,way/4272878,4272878.0,way,None,None,None,None,None,None,None,None,concrete,None,5.920195,dedicated,NaN,NaN,0,5.920195,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",000000,5.920195,False,True,4,1,0.114063,0.114063,1
1,-5781459007026747408,8604236943511184175,000001,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,50.000000,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",000001,50.000000,False,True,3,1,0.114063,0.114063,1
2,8604236943511184175,-1121181787614706977,000002,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,50.000000,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",000002,50.000000,False,False,2,0,0.100000,0.100000,1
3,-1121181787614706977,7016047240835109748,000003,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,50.000000,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",000003,50.000000,False,False,2,0,0.100000,0.100000,1
4,7016047240835109748,-7152654980191551096,000004,walk_track,OSM_highway_dedicated,track,way/4297439,4297439.0,way,None,None,Chemin du Milieu,None,None,None,None,grade2,None,None,701.230056,dedicated,NaN,NaN,1,50.000000,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",000004,50.000000,False,False,2,0,0.100000,0.100000,1


In [7]:
save(save_filtered_attributes, row, segmented_net_index.to_crs(target_crs)[["segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]], attribute)

Geometries cleaned
Filtered data saved for attribute: connectivite in format: parquet
Filtered data saved for attribute: connectivite in format: csv
Filtered data saved for attribute: connectivite in format: gpkg


In [8]:
segmented_net_index.to_crs(target_crs)[["key","segment_id","geometry","conn_branching_in_buffer","conn_index_score","filtered"]]

,key,segment_id,geometry,conn_branching_in_buffer,conn_index_score,filtered
0,000000,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",0.114063,0.114063,1
1,000001,000001,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",0.114063,0.114063,1
2,000002,000002,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",0.100000,0.100000,1
3,000003,000003,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",0.100000,0.100000,1
4,000004,000004,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",0.100000,0.100000,1
...,...,...,...,...,...,...
555370,555370,555370,"LINESTRING (6.12008 46.22094, 6.12017 46.22088)",0.000000,0.156250,1
555371,555371,555371,"LINESTRING (6.12017 46.22088, 6.12041 46.22071...",0.212500,0.212500,1
555372,555372,555372,"LINESTRING (6.12003 46.22091, 6.12009 46.22087)",0.000000,0.170313,1
555373,555373,555373,"LINESTRING (6.12009 46.22087, 6.12036 46.22068...",0.212500,0.212500,1


## Pente

In [10]:
# Initialize
attribute = None
row = None
segmented_net_slope = None
print("Data initialized")

# Load
###----change here-----
attribute = 'pente'
###--------------------
row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
print(f"Processing attribute: {attribute}")

source_path_col = f"source_path_{territory}"
file_name_col = f"file_name_{territory}"
raster_path = f"{input_file_path}/{row[source_path_col]}/{row[file_name_col]}"

segmented_net_slope = compute_segment_slope(
    segmented_net,
    raster_path,
    crs_meter_epsg=2056,
    length_col="length",
    smooth_sigma=1.0,
    smooth_hops=2,
)
segmented_net_slope['pente'] = segmented_net_slope['slope_pct_abs_smoothed']
segmented_net_slope['pente_brute'] = segmented_net_slope['slope_pct_abs']
segmented_net_slope['filtered'] = segmented_net_slope['pente'].notna().astype(int)

print("\nDescriptive statistics (pente lissée):")
print(segmented_net_slope['pente'].describe())

segmented_net_slope.to_crs(target_crs)[[
    'segment_id',
    'geometry',
    'pente',
    'pente_brute',
    'elev_start_m',
    'elev_end_m',
    'elev_delta_m',
    'slope_pct_signed',
    'filtered',
]].head()

Data initialized
Processing attribute: pente

Descriptive statistics (pente lissée):
count    555375.000000
mean          5.650841
std           6.146237
min           0.000000
25%           2.263555
50%           4.041330
75%           7.105228
max         381.751411
Name: pente, dtype: float64


,segment_id,geometry,pente,pente_brute,elev_start_m,elev_end_m,elev_delta_m,slope_pct_signed,filtered
0,000000,"LINESTRING (6.1215 46.20185, 6.12157 46.20187)",2.057334,0.000000,14.854813,14.854813,0.000000,0.000000,1
1,000001,"LINESTRING (6.08229 46.30288, 6.08185 46.3032)",0.602740,0.081681,2.104920,2.064079,-0.040841,-0.081681,1
2,000002,"LINESTRING (6.08185 46.3032, 6.0814 46.30353)",1.709063,1.822778,2.064079,2.975468,0.911389,1.822778,1
3,000003,"LINESTRING (6.0814 46.30353, 6.08095 46.30385)",2.371369,4.104128,2.975468,0.923404,-2.052064,-4.104128,1
4,000004,"LINESTRING (6.08095 46.30385, 6.08051 46.30418)",1.842500,0.764747,0.923404,1.305777,0.382373,0.764747,1


In [11]:
save(
    save_filtered_attributes,
    row,
    segmented_net_slope.to_crs(target_crs)[[
        'segment_id',
        'geometry',
        'pente',
        'pente_brute',
        'elev_start_m',
        'elev_end_m',
        'elev_delta_m',
        'slope_pct_signed',
        'slope_pct_abs',
        'slope_pct_abs_smoothed',
        'filtered',
    ]],
    attribute,
 )

Geometries cleaned
Filtered data saved for attribute: pente in format: parquet
Filtered data saved for attribute: pente in format: csv
Filtered data saved for attribute: pente in format: gpkg


## Vitesse

In [12]:
walk_edges_graph_path = f"{input_file_path}/networkGG/walk_edges_graph.geojson"
walk_edges_graph = gpd.read_file(walk_edges_graph_path)

import json
import re
import numpy as np
import pandas as pd
import geopandas as gpd
from pathlib import Path

# --- 1) Parse maxspeed -------------------------------------------------------
def parse_maxspeed(v):
    if v is None or pd.isna(v):
        return np.nan
    if isinstance(v, (int, float)):
        return float(v)

    s = str(v).lower().strip()
    if s in {"walk", "walking", "foot"}:
        return 5.0
    if s in {"signals", "none", "variable"}:
        return np.nan

    if "mph" in s:
        m = re.search(r"(\d+)", s)
        return float(m.group(1)) * 1.60934 if m else np.nan

    m = re.search(r"(\d+)", s)
    return float(m.group(1)) if m else np.nan


# --- 2) Highway normalization (list -> str) ----------------------------------
def normalize_highway(v):
    if v is None or pd.isna(v):
        return np.nan
    if isinstance(v, (list, tuple, set)):
        return str(list(v)[0]) if len(v) else np.nan
    return str(v)


# --- 3) Default speeds (CH/urban Europe, adjust if you want) -----------------
DEFAULT_SPEED_BY_HIGHWAY = {
    "motorway": 120, "trunk": 80,
    "primary": 50, "secondary": 50, "tertiary": 50,
    "primary_link": 50, "secondary_link": 50, "tertiary_link": 50,
    "residential": 30, "unclassified": 30,
    "service": 20, "living_street": 20,

    # pedestrian-ish
    "pedestrian": 10,
    "footway": 5, "path": 5, "steps": 5,
    "track": 30,
    "platform": 5, "corridor": 5,
}

ZONE_MAXSPEED_MAP = {
    "CH:urban": 50,
    "CH:rural": 80,
    "CH:motorway": 120,
}


# --- 4) Compute speed_kph on a COPY only ------------------------------------
def compute_speed_kph(df: gpd.GeoDataFrame) -> pd.Series:
    speed = pd.Series(np.nan, index=df.index, dtype="float64")

    # a) use explicit maxspeed tags if present
    for col in ["maxspeed:forward", "maxspeed:backward", "maxspeed"]:
        if col in df.columns:
            speed = speed.fillna(df[col].apply(parse_maxspeed))

    # b) zone-based hints
    if "zone:maxspeed" in df.columns:
        z = df["zone:maxspeed"].astype(str)
        speed = speed.fillna(z.map(ZONE_MAXSPEED_MAP))

    # c) highway fallback
    if "highway" in df.columns:
        hw = df["highway"].apply(normalize_highway)
        speed = speed.fillna(hw.map(DEFAULT_SPEED_BY_HIGHWAY))

    # d) final fallback
    speed = speed.fillna(30.0)

    return speed


# --- 5) Build output gdf and export -----------------------------------------
def export_speed_layer(edges_gdf_graph: gpd.GeoDataFrame, out_dir: Path, basename="edges_speed"):
    out_dir.mkdir(parents=True, exist_ok=True)

    out = edges_gdf_graph.copy()
    out["highway_norm"] = out["highway"].apply(normalize_highway) if "highway" in out.columns else np.nan
    # fill missing highway (optional, conservative)
    out["highway_norm"] = out["highway_norm"].fillna("service")

    out["speed_kph"] = compute_speed_kph(out)

    # keep only useful columns
    keep = [c for c in ["edge_id", "u", "v", "key", "osm_id", "id", "element", "name", "highway", "highway_norm",
                        "maxspeed", "zone:maxspeed", "maxspeed:type", "speed_kph", "length", "len_m", "geometry"]
            if c in out.columns]
    out = out[keep].copy()

    # GeoJSON
    geojson_path = out_dir / f"{basename}.geojson"
    out.to_file(geojson_path, driver="GeoJSON")

    # Parquet (geoparquet)
    parquet_path = out_dir / f"{basename}.parquet"
    out.to_parquet(parquet_path, index=False)

    print("Exported:", geojson_path)
    print("Exported:", parquet_path)
    print("Missing speed_kph:", int(out["speed_kph"].isna().sum()))
    return out



out_dir = Path(f"{input_file_path}/attributs/GG/vitesse")  # choose where you want
speed_gdf = export_speed_layer(walk_edges_graph, out_dir, basename=f"vitesse_all_edges")


Exported: ../../Data/input/attributs/GG/vitesse/vitesse_all_edges.geojson
Exported: ../../Data/input/attributs/GG/vitesse/vitesse_all_edges.parquet
Missing speed_kph: 0


## Merge files

In [13]:
#Helper

territories = ['GE', 'FR', 'VD']

# ── Chargement générique avec gestion des valeurs manquantes ──
def load_territory(attributs_info, attribute, territory, input_file_path, target_crs):
    row = attributs_info[attributs_info['attribute'] == attribute].iloc[0]
    source_path = row.get(f'source_path_{territory}')
    file_name   = row.get(f'file_name_{territory}')

    # Cas manquants : cellule vide, NaN, ou chemin inexistant
    if pd.isna(source_path) or pd.isna(file_name):
        print(f"[{territory}] Pas de fichier défini: territoire ignoré.")
        return None

    full_path = f"{input_file_path}/{source_path}/{file_name}"

    if not os.path.exists(full_path):
        print(f"[{territory}] Fichier introuvable : {full_path} territoire ignoré.")
        return None

    gdf = gpd.read_file(full_path)
    return gdf.to_crs(target_crs)

### Sens inverse

In [14]:
# ── CAHNGE HERE ──────────────────────────────────────────────
attribute = 'sens_inverse'
#──────────────────────────────────────────────

# ── Chargement de tous les territoires ────────────────────────
gdfs = {}
for territory in territories:
    result = load_territory(
        attributs_info, attribute, territory, input_file_path, target_crs
    )
    if result is not None:
        result['territory'] = territory
        gdfs[territory] = result

# ── Merge uniquement si au moins un territoire chargé ─────────
if gdfs:
    attribute_merged = gpd.GeoDataFrame(
        pd.concat(gdfs.values(), ignore_index=True),
        crs=target_crs
    )
    print(f"Merge réussi : {len(attribute_merged)} lignes — territoires : {list(gdfs.keys())}")
else:
    print(f"Aucun fichier trouvé pour l'attribut '{attribute}'.")
    attribute_merged = None


#Save merged file into input/attributs/GG/filename

if attribute_merged is not None:
    output_dir  = os.path.join(input_file_path, 'attributs', 'GG', attribute)
    output_file = os.path.join(output_dir, f"{attribute}.gpkg")
    
    os.makedirs(output_dir, exist_ok=True)
    attribute_merged.to_file(output_file, driver='GPKG')
    
    print(f"Fichier sauvegardé : {output_file}")
else:
    print(f"Aucun fichier à sauvegarder pour '{attribute}'.")

IndexError: single positional indexer is out-of-bounds

### Tourner à droite ###

In [ ]:
# ── CHANGE HERE ──────────────────────────────────────────────
attribute = 'tourner_droite'
#──────────────────────────────────────────────

# ── Chargement de tous les territoires ────────────────────────
gdfs = {}
for territory in territories:
    result = load_territory(
        attributs_info, attribute, territory, input_file_path, target_crs
    )
    if result is not None:
        result['territory'] = territory
        gdfs[territory] = result

# ── Merge uniquement si au moins un territoire chargé ─────────
if gdfs:
    attribute_merged = gpd.GeoDataFrame(
        pd.concat(gdfs.values(), ignore_index=True),
        crs=target_crs
    )
    print(f"Merge réussi : {len(attribute_merged)} lignes — territoires : {list(gdfs.keys())}")
else:
    print(f"Aucun fichier trouvé pour l'attribut '{attribute}'.")
    attribute_merged = None


#Save merged file into input/attributs/GG/filename

if attribute_merged is not None:
    output_dir  = os.path.join(input_file_path, 'attributs', 'GG', attribute)
    output_file = os.path.join(output_dir, f"{attribute}.gpkg")
    
    os.makedirs(output_dir, exist_ok=True)
    attribute_merged.to_file(output_file, driver='GPKG')
    
    print(f"Fichier sauvegardé : {output_file}")
else:
    print(f"Aucun fichier à sauvegarder pour '{attribute}'.")

IndexError: single positional indexer is out-of-bounds